# Preparation

In [ ]:
"""Data Name"""

# Regression
# dataName = "IMDBCLinear"
# dataName = "IMDBLargeCLinear"
# dataName = "taxi"
# dataName = "stackn"

# Classification
# dataName = "IMDBC5"
# dataName = "IMDBLargeC5"
dataName = "Brazilnew"

In [ ]:
BASE_DICT = {
    "IMDBC5":"IMDBC5base",
    "IMDBLargeC5":"IMDBC5base",
    "Brazilnew":"Brazilbase",
    "IMDBCLinear": "IMDBCLinearbase",
    "IMDBLargeCLinear": "IMDBCLinearbase",
    "stackn":"stackbase",
    "taxi":"taxibase"
}

In [ ]:
SAMPLE_PROP_DICT = {
    "IMDBC5":0.0128,
    "IMDBLargeC5":0.0016,
    "Brazilnew":0.0016,
    "IMDBCLinear": 0.0032,
    "IMDBLargeCLinear": 0.0016,
    "stackn":0.0032,
    "taxi":0.0032
}

In [ ]:
baseName = BASE_DICT[dataName]
PROP = SAMPLE_PROP_DICT[dataName]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import os
import lightgbm as lgb
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

In [ ]:
X_train = []
X_test = []
y_train = []
y_test = []

## Load Data

In [ ]:
def LoadDataset(dataset):
    global X_train, X_test, X_val, y_val,y_train, y_test
    DATASET_DIR = '/home/jiayi/disk/C-craig/dataset/'
    
    X_train = np.load(DATASET_DIR + "{}-train-X.npy".format(dataset))
    X_val = np.load(DATASET_DIR + "{}-val-X.npy".format(dataset))
    X_test = np.load(DATASET_DIR + "{}-test-X.npy".format(dataset))

    y_train = np.load(DATASET_DIR + "{}-train-y.npy".format(dataset))
    y_val = np.load(DATASET_DIR + "{}-val-y.npy".format(dataset))
    y_test = np.load(DATASET_DIR + "{}-test-y.npy".format(dataset))

In [ ]:
# LoadDataset("Brazil5")
# LoadDataset("IMDBLargeC10")
# LoadDataset("IMDBC10")
# LoadDataset("stackLinear")
# LoadDataset("taxi")
# LoadDataset("taxi-single")
# LoadDataset("stackn-single")
# LoadDataset("stackn")
# LoadDataset("IMDBC5base")
# LoadDataset("IMDBCLinear")
# LoadDataset("Brazilnew")
# LoadDataset("IMDBC5base")

## Load Predict Result

In [ ]:
def getPredictions(data, greedy, coreset_from, subset_size, itr=0):
    DIR = '/home/jiayi/disk/C-craig/predict/'
    file = '{}-greedy-{}-{}-{}-predict-itr-{}.npy'.format(data, greedy, coreset_from,subset_size, itr)
    # np.save(DIR+file, best_predict )
    my_predictions = np.load(DIR+file)
    return my_predictions

## Load test DF

In [ ]:
def getTestDF():
    DIR = '/home/jiayi/disk/C-craig/dataset/'
    file = '{}-test.csv'.format(dataName)

    if os.path.exists(DIR + file) ==False:
        file = file = '{}-test-X.csv'.format(dataName)

    testDF = pd.read_csv(DIR + file)
    return testDF

# Regression

## Utility Functions

### T+ to base

In [ ]:
def PredToBaseReg(tdf, my_predictions):
    """ Aggregate prediction results in T+ to base """
    tdf['predict'] = my_predictions
    
    if dataName in ['IMDBCLinear', 'IMDBLargeCLinear']:
        z = tdf.groupby('movie_id')
    elif dataName in ['taxi']:
        z = tdf.groupby('f642')
    elif dataName in ['stackn']:
        z = tdf.groupby('newUid')

    predict_group = z['predict'].mean() 
    
    if dataName in ['IMDBCLinear', 'IMDBLargeCLinear']:
        real_group    = z['rating'].mean()
    elif dataName in ['taxi']:
        real_group    = z['target'].mean()
    elif dataName in ['stackn']:
        real_group    = z['reputation'].mean()
    
    MSE = mean_squared_error(predict_group, real_group)
    RMSE = np.sqrt(MSE)
    print("Mean Squared     Error : " + str(mean_squared_error(predict_group, real_group)))
    print("RMSE is : ", RMSE)
    return RMSE

### Evaluate on base table

In [ ]:
def EvaluateReg():
    
    tdf = getTestDF()
    
    # Full
    fullPred = getPredictions(dataName, greedy=0, coreset_from="diskOurs", subset_size =1.0)
    fullRMSE = PredToBaseReg(tdf.copy(), fullPred)
    
    # Sample-Join
    samplePred = getPredictions(dataName, greedy=0, coreset_from="diskOurs", subset_size =PROP)
    sampleRMSE = PredToBaseReg(tdf.copy(), samplePred)
    
    # Join-Coreset
    JCPred = getPredictions(dataName, greedy=0, coreset_from="diskJC", subset_size =PROP)
    JCRMSE = PredToBaseReg(tdf.copy(), samplePred)
    
    # Coreset-Join
    CJPred = getPredictions(dataName, greedy=0, coreset_from="diskCJ", subset_size =PROP)
    CJRMSE = PredToBaseReg(tdf.copy(), samplePred)
    
    # Ours
    oursPred = getPredictions(dataName, greedy=1, coreset_from="diskOurs", subset_size =PROP)
    oursRMSE = PredToBaseReg(tdf.copy(), oursPred)
    
    print("Full   :", fullRMSE)
    print("Sample :", sampleRMSE)
    print("JC:", sampleRMSE)
    print("CJ   :", oursRMSE)
    print("Ours   :", oursRMSE)
    
    return [fullRMSE, sampleRMSE, JCRMSE, JCRMSE oursRMSE]

## Run

In [ ]:
# EvaluateReg()

# Classification

## Utility Functions

### T+ to base

In [ ]:
def PredToBaseClass(tdf, my_predictions):

    my_predictions = np.argmax(my_predictions, axis=1)
    tdf['predict'] = my_predictions
    

    if dataName in ['IMDBC5', 'IMDBLargeC5']:
        z = tdf.groupby('movie_id')
    elif dataName == 'Brazilnew':
        z = tdf.groupby('review_id')
        

    if dataName in ['IMDBC5', 'IMDBLargeC5']:
        real_group    = z['rating'].mean()
    elif dataName == 'Brazilnew':
        real_group = z['review_score'].mean()
    
    acc = metrics.accuracy_score(predict_group.predict, real_group)
    print("Acc is ", acc)
    return acc

### Evaluate on base table

In [ ]:
def EvaluateClass():
    tdf = getTestDF()
    
    # Full
    fullPred = getPredictions(dataName, greedy=0, coreset_from="diskOurs", subset_size =1.0)
    fullAcc = PredToBaseClass(tdf.copy(), fullPred)
    
    # Sample-join
    samplePred = getPredictions(dataName, greedy=0, coreset_from="diskOurs", subset_size =PROP)
    sampleAcc = PredToBaseClass(tdf.copy(), samplePred)
    
    # Join-Coreset
    JCPred = getPredictions(dataName, greedy=1, coreset_from="diskJC", subset_size =PROP)
    JCAcc = PredToBaseClass(tdf.copy(), oursPred)
    
    # Coreset-Join
    CJPred = getPredictions(dataName, greedy=1, coreset_from="diskJC", subset_size =PROP)
    CJAcc = PredToBaseClass(tdf.copy(), oursPred)
    
    # Ours
    oursPred = getPredictions(dataName, greedy=1, coreset_from="diskOurs", subset_size =PROP)
    oursAcc = PredToBaseClass(tdf.copy(), oursPred)
    
    print("Full   :", fullAcc
    print("Sample :", sampleAcc)
    print("JC   :", JCAcc)
    print("CJ   :", CJAcc)
    print("Ours   :", oursAcc)
    
    return [fullAcc, sampleAcc, JCAcc, CJAcc, oursAcc]

## Run

In [ ]:
EvaluateClass()